# 🪐 Crater Segmentation — Ghost-RDT-UNet++ (Novel Architecture)

**Project:** CS776 — Deep Learning for Computer Vision, IIT Kanpur

## What is Ghost-RDT-UNet++?

This is a **new hybrid architecture** that fuses two ideas:

| Source | What we take |
|---|---|
| **RDT-UNet++** | Dilated Residual Dense encoder, skip attention hierarchy (CBAM/Window/Cross/Circular Transformer), multi-scale fusion head, deep supervision |
| **Ghost Convolution** | `ResidualGhostBlock` replaces every `DoubleConv` in the decoder; ECA (Efficient Channel Attention) after every decoder stage |

### Why this combination works

The RDT-UNet++ encoder is already rich — it produces multi-scale dilated features with transformer-quality skip attention. The **bottleneck** in the original design is the decoder: it uses standard `ConvBnRelu` blocks that are computationally heavier than needed, because the decoder's job is feature reconstruction, not feature discovery.

**Ghost Convolution** solves this perfectly:
- It generates **half** the features with a standard 3×3 conv (`primary_conv`) and **hallucinates** the other half with a cheap depthwise conv (`cheap_operation`) — same channel count, ~50% fewer parameters.
- Stacking two of these as a `ResidualGhostBlock` (with a residual shortcut) gives the decoder equivalent representational power to a DoubleConv at roughly half the cost.
- `ECA_Layer` (Efficient Channel Attention) adds practically free global channel re-weighting after each decoder stage (only ~3–5 parameters), helping suppress irrelevant upsampled channels.

### Architecture Overview

```
Input (1×512×512)
    ↓ Stem (ConvBnRelu × 2)
    ↓ DilatedResidualDenseBlock × 4 (Encoder — rates 1,2,4)
       ├── Skip L1: CBAMSkip          (full-res, 32ch)
       ├── Skip L2: CBAMSkip          (H/2, 64ch)
       ├── Skip L3: CrossScaleTransformerSkip (window self-attn + pooled cross-attn)
       └── Skip L4: CircularTransformerSkip  (window attn + radial geometry gate)
    ↓ Bottleneck (DilatedResidualDenseBlock)
    ↓ Decoder × 4 (NEW: GhostRDTUpBlock)
       └── ResidualGhostBlock × 2 + ECA after each decoder stage
    ↓ ChannelReducedMSFHead (d1 + d2_32ch + d3_32ch → logit)
    ↓ AuxHead at d2, d3 (deep supervision — train only)
```

### Ghost Convolution — Key Insight
```
Standard DoubleConv:  [3×3 conv → BN → ReLU] × 2   → C_out channels
ResidualGhostBlock:   GhostModule × 2 + residual shortcut
    GhostModule:  primary_conv(3×3, C_in→C_out/2)  → x1  (half features, standard)
                  cheap_op(3×3 DW, C_out/2→C_out/2) → x2  (other half, depthwise)
                  output = cat(x1, x2)               → C_out features
Cost saving: ~50% fewer FLOPs in decoder vs original RDT-UNet++
```

---
## Pipeline
```
Input Images → Tiling + Copy-Paste Augmentation (holdout separated)
→ CraterDataset with online augmentation
→ Ghost-RDT-UNet++ forward pass
→ DeepSupervisionLoss (BCE + Dice + aux heads)
→ AMP training + CosineAnnealingLR
→ Test evaluation + Holdout demo
```

---
## Section 0 — Global Config

In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

MODEL_NAME         = 'ghost_rdt_unet_plusplus'
HOLDOUT_IMAGE_NAME = 'thm_dir_N00_330.png'   # ← change to your hold-out filename

print(f'✅ Model          : {MODEL_NAME}')
print(f'✅ Hold-out image : {HOLDOUT_IMAGE_NAME}')

---
## Section 1 — Setup & Imports

In [ ]:
!pip install torch torchvision opencv-python tqdm matplotlib numpy --quiet

In [ ]:
import os, sys, json, time, random, shutil, math, warnings
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from torch.cuda.amp import GradScaler, autocast

warnings.filterwarnings('ignore')

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = torch.cuda.is_available()
print(f'✓ Device    : {DEVICE}')
print(f'✓ PyTorch   : {torch.__version__}')
print(f'✓ CUDA      : {torch.cuda.is_available()}')
print(f'✓ AMP (fp16): {USE_AMP}')
print(f'✓ Model     : {MODEL_NAME}')

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(42)
print('✓ Seed set to 42')

---
## Section 2 — Training Configuration

In [ ]:
@dataclass
class TrainConfig:
    tile_image_dir    : str   = './tiles/images'
    tile_mask_dir     : str   = './tiles/masks'
    holdout_image_dir : str   = './tiles_holdout/images'
    holdout_mask_dir  : str   = './tiles_holdout/masks'
    output_dir        : str   = './runs'
    train_ratio       : float = 0.70
    val_ratio         : float = 0.15
    # test = remainder (0.15)
    epochs            : int   = 45
    batch_size        : int   = 4
    learning_rate     : float = 1e-4
    weight_decay      : float = 1e-5
    early_stop_patience: int  = 10
    num_workers       : int   = 2
    pin_memory        : bool  = True
    bce_weight        : float = 0.5
    dice_weight       : float = 0.5
    pos_weight        : float = 10.0
    in_channels       : int   = 1
    out_channels      : int   = 1
    seed              : int   = 42
    model_name        : str   = MODEL_NAME

CFG = TrainConfig()
print(f'✓ Config loaded  →  model={CFG.model_name}')
print(f'   epochs={CFG.epochs}  batch={CFG.batch_size}  lr={CFG.learning_rate}')

---
## Section 3 — Tiling + Copy-Paste Augmentation

> **Update the paths below to match your system before running.**  
> Skip this section (don't run `run_tiling()`) if tiles are already generated.  
> Two passes: 23 training images → `./tiles/` | 1 hold-out image → `./tiles_holdout/`

In [ ]:
# ── Tiling paths ──────────────────────────────────────────────────────────────
IMAGE_DIR = r"/kaggle/input/datasets/sahibjparmar/mars-crater-dataset-for-semantic-segmentation/Dataset/Images"
MASK_DIR  = r"/kaggle/input/datasets/sahibjparmar/mars-crater-dataset-for-semantic-segmentation/Dataset/Mars_Crater_Segmentation_Dataset_02-32_km_radius/edge_thicker"

OUTPUT_IMAGE_DIR         = './tiles/images'
OUTPUT_MASK_DIR          = './tiles/masks'
OUTPUT_HOLDOUT_IMAGE_DIR = './tiles_holdout/images'
OUTPUT_HOLDOUT_MASK_DIR  = './tiles_holdout/masks'

TILE_SIZE             = 512
STRIDE                = 512
ENABLE_COPY_PASTE     = True
PASTES_PER_IMAGE      = 3
AUGMENTED_COPIES      = 2
MIN_CRATER_PIXELS     = 30
BLEND_FEATHER_RADIUS  = 5
MAX_PASTE_ATTEMPTS    = 10
BLACK_STRIP_THRESHOLD = 0.15
BLACK_PIXEL_VALUE     = 10

for d in [OUTPUT_IMAGE_DIR, OUTPUT_MASK_DIR,
          OUTPUT_HOLDOUT_IMAGE_DIR, OUTPUT_HOLDOUT_MASK_DIR]:
    os.makedirs(d, exist_ok=True)
print('✓ Output directories ready')

In [ ]:
# ── Artifact Detection ────────────────────────────────────────────────────────
def has_black_strip(image: np.ndarray, threshold=BLACK_STRIP_THRESHOLD) -> bool:
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if image.ndim == 3 else image
    return (gray < BLACK_PIXEL_VALUE).sum() / gray.size > threshold

# ── Crater Instance Library ───────────────────────────────────────────────────
class CraterLibrary:
    def __init__(self):
        self.craters = []

    def extract_from_tile(self, image: np.ndarray, mask: np.ndarray):
        _, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
        for i in range(1, len(stats)):
            x, y, w, h, area = stats[i]
            if area < MIN_CRATER_PIXELS: continue
            pad = max(5, int(max(w, h) * 0.1))
            x1, y1 = max(0, x-pad), max(0, y-pad)
            x2, y2 = min(image.shape[1], x+w+pad), min(image.shape[0], y+h+pad)
            self.craters.append({'img':  image[y1:y2, x1:x2].copy(),
                                 'mask': mask[y1:y2,  x1:x2].copy(),
                                 'radius': int(max(w, h) / 2), 'area': area})

    def sample(self, prefer_small=True):
        if not self.craters: return None
        if prefer_small and random.random() < 0.65:
            radii   = np.array([c['radius'] for c in self.craters], dtype=float)
            weights = 1.0 / (radii + 1e-6); weights /= weights.sum()
            idx = np.random.choice(len(self.craters), p=weights)
        else:
            idx = random.randint(0, len(self.craters) - 1)
        return self.craters[idx]

    def __len__(self): return len(self.craters)

# ── Copy-Paste Augmentation ───────────────────────────────────────────────────
def gaussian_feather_mask(mask_patch, radius):
    blurred = cv2.GaussianBlur(mask_patch.astype(np.float32),
                               (2*radius+1, 2*radius+1), sigmaX=radius)
    return np.clip(blurred / (blurred.max() + 1e-6), 0, 1)

def paste_overlaps_existing(target_mask, y1, x1, ch, cw):
    return (target_mask[y1:y1+ch, x1:x1+cw] > 0).any()

def copy_paste_augment(target_img, target_mask, library, n_pastes=PASTES_PER_IMAGE):
    aug_img  = target_img.copy().astype(np.float32)
    aug_mask = target_mask.copy()
    H, W     = aug_img.shape[:2]
    for _ in range(n_pastes):
        crater = library.sample(prefer_small=True)
        if crater is None: break
        c_img, c_mask = crater['img'], crater['mask']
        ch, cw = c_img.shape[:2]
        if ch >= H or cw >= W: continue
        placed = False
        for _ in range(MAX_PASTE_ATTEMPTS):
            y1 = random.randint(0, H - ch); x1 = random.randint(0, W - cw)
            if not paste_overlaps_existing(aug_mask, y1, x1, ch, cw):
                placed = True; break
        if not placed: continue
        weight = gaussian_feather_mask(c_mask, BLEND_FEATHER_RADIUS)
        if aug_img.ndim == 3 and c_img.ndim == 2: c_img = cv2.cvtColor(c_img, cv2.COLOR_GRAY2BGR)
        elif aug_img.ndim == 2 and c_img.ndim == 3: c_img = cv2.cvtColor(c_img, cv2.COLOR_BGR2GRAY)
        if aug_img.ndim == 3:
            w3 = weight[:, :, np.newaxis]
            aug_img[y1:y1+ch, x1:x1+cw] = (
                w3*c_img.astype(np.float32) + (1-w3)*aug_img[y1:y1+ch, x1:x1+cw])
        else:
            aug_img[y1:y1+ch, x1:x1+cw] = (
                weight*c_img.astype(np.float32) + (1-weight)*aug_img[y1:y1+ch, x1:x1+cw])
        aug_mask[y1:y1+ch, x1:x1+cw] = np.maximum(aug_mask[y1:y1+ch, x1:x1+cw], c_mask)
    return aug_img.astype(np.uint8), aug_mask

# ── Tiling helpers ────────────────────────────────────────────────────────────
def get_base_name(filename): return os.path.splitext(filename)[0]
def find_mask(image_base, mask_files):
    for m in mask_files:
        if image_base in m: return m
    return None

def tile_one_image(image_path, mask_path, base_name, out_img, out_msk,
                   library=None, enable_augment=False):
    image = cv2.imread(image_path); mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if image is None or mask is None: return 0, 0
    h, w = image.shape[:2]; orig = aug = 0
    for y in range(0, h - TILE_SIZE + 1, STRIDE):
        for x in range(0, w - TILE_SIZE + 1, STRIDE):
            ip = image[y:y+TILE_SIZE, x:x+TILE_SIZE]
            mp = mask[y:y+TILE_SIZE,  x:x+TILE_SIZE]
            if (mp > 0).sum() < 10 or has_black_strip(ip): continue
            name = f'{base_name}_x{x}_y{y}.png'
            cv2.imwrite(os.path.join(out_img, name), ip)
            cv2.imwrite(os.path.join(out_msk, name), mp)
            orig += 1
            if library is not None: library.extract_from_tile(ip, mp)
            if enable_augment and library is not None and len(library) > 5:
                for ai in range(AUGMENTED_COPIES):
                    ai_img, ai_msk = copy_paste_augment(ip, mp, library)
                    if (ai_msk > 0).sum() < 10: continue
                    cv2.imwrite(os.path.join(out_img, f'{base_name}_x{x}_y{y}_aug{ai}.png'), ai_img)
                    cv2.imwrite(os.path.join(out_msk, f'{base_name}_x{x}_y{y}_aug{ai}.png'), ai_msk)
                    aug += 1
    return orig, aug

def run_tiling():
    image_files = os.listdir(IMAGE_DIR); mask_files = os.listdir(MASK_DIR)
    library = CraterLibrary(); total_orig = total_aug = skipped = 0
    holdout_done = False

    for img_file in tqdm(image_files, desc='Tiling images'):
        base = get_base_name(img_file); mask_file = find_mask(base, mask_files)
        if mask_file is None: skipped += 1; continue

        # Separate hold-out image from training tiles
        if HOLDOUT_IMAGE_NAME and base in HOLDOUT_IMAGE_NAME:
            orig, _ = tile_one_image(
                os.path.join(IMAGE_DIR, img_file),
                os.path.join(MASK_DIR,  mask_file), base,
                OUTPUT_HOLDOUT_IMAGE_DIR, OUTPUT_HOLDOUT_MASK_DIR,
                library=None, enable_augment=False)
            print(f'  Hold-out: {img_file} → {orig} tiles (no augmentation)')
            holdout_done = True
            continue

        orig, aug = tile_one_image(
            os.path.join(IMAGE_DIR, img_file),
            os.path.join(MASK_DIR,  mask_file), base,
            OUTPUT_IMAGE_DIR, OUTPUT_MASK_DIR,
            library=library, enable_augment=ENABLE_COPY_PASTE)
        total_orig += orig; total_aug += aug

    print(f'\n{"="*50}')
    print(f'  Training tiles (orig)  : {total_orig:,}')
    print(f'  Training tiles (aug)   : {total_aug:,}')
    print(f'  Total training tiles   : {total_orig + total_aug:,}')
    print(f'  Crater library         : {len(library):,} instances')
    print(f'  Hold-out tiled         : {holdout_done}')
    print(f'{"="*50}')

# ▶ Run tiling — comment out if tiles already exist
run_tiling()

---
## Section 4 — Dataset & DataLoaders

In [ ]:
class CraterDataset(Dataset):
    """Reads pre-tiled images and binary rim masks."""
    def __init__(self, image_dir, mask_dir, augment=False):
        self.image_dir = Path(image_dir)
        self.mask_dir  = Path(mask_dir)
        self.augment   = augment
        self.files     = sorted([f.name for f in self.image_dir.glob('*.png')])
        if not self.files:
            raise RuntimeError(f'No PNG tiles found in {image_dir}')
        print(f'  Dataset: {len(self.files)} tiles  (augment={augment})')

    def _augment(self, img, mask):
        # Horizontal flip
        if random.random() > 0.5:
            img = cv2.flip(img, 1); mask = cv2.flip(mask, 1)
        # Vertical flip
        if random.random() > 0.5:
            img = cv2.flip(img, 0); mask = cv2.flip(mask, 0)
        # 90° rotation
        k = random.randint(0, 3)
        if k > 0:
            img = np.rot90(img, k).copy(); mask = np.rot90(mask, k).copy()
        # Brightness / contrast jitter
        alpha = random.uniform(0.8, 1.2)   # contrast
        beta  = random.uniform(-15, 15)    # brightness
        img   = np.clip(img.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
        # Gaussian blur (occasional)
        if random.random() > 0.8:
            img = cv2.GaussianBlur(img, (3, 3), 0)
        return img, mask

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        name = self.files[idx]
        img  = cv2.imread(str(self.image_dir / name), cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(str(self.mask_dir  / name), cv2.IMREAD_GRAYSCALE)
        if img is None or mask is None:
            raise FileNotFoundError(f'Cannot read tile: {name}')
        if self.augment: img, mask = self._augment(img, mask)
        img_t  = torch.from_numpy(img.astype(np.float32)  / 255.0).unsqueeze(0)
        mask_t = torch.from_numpy((mask > 127).astype(np.float32)).unsqueeze(0)
        return img_t, mask_t


def make_dataloaders(cfg):
    full_ds = CraterDataset(cfg.tile_image_dir, cfg.tile_mask_dir, augment=False)
    n       = len(full_ds)
    n_train = int(n * cfg.train_ratio)
    n_val   = int(n * cfg.val_ratio)
    n_test  = n - n_train - n_val

    indices  = list(range(n))
    random.shuffle(indices)
    train_idx = indices[:n_train]
    val_idx   = indices[n_train:n_train + n_val]
    test_idx  = indices[n_train + n_val:]

    # Training split gets online augmentation
    train_ds = CraterDataset(cfg.tile_image_dir, cfg.tile_mask_dir, augment=True)
    train_loader = DataLoader(Subset(train_ds, train_idx), batch_size=cfg.batch_size,
                              shuffle=True,  num_workers=cfg.num_workers,
                              pin_memory=cfg.pin_memory, drop_last=True)
    val_loader   = DataLoader(Subset(full_ds, val_idx),   batch_size=cfg.batch_size,
                              shuffle=False, num_workers=cfg.num_workers,
                              pin_memory=cfg.pin_memory)
    test_loader  = DataLoader(Subset(full_ds, test_idx),  batch_size=cfg.batch_size,
                              shuffle=False, num_workers=cfg.num_workers,
                              pin_memory=cfg.pin_memory)

    print(f'\n  Split → train:{n_train}  val:{n_val}  test:{n_test}')
    return train_loader, val_loader, test_loader, test_idx


def make_holdout_loader(cfg):
    ds = CraterDataset(cfg.holdout_image_dir, cfg.holdout_mask_dir, augment=False)
    return DataLoader(ds, batch_size=1, shuffle=False,
                      num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)

print('✓ Dataset and DataLoader helpers defined')

---
## Section 5 — Ghost-RDT-UNet++ Architecture

### 5a. Shared Primitive Blocks

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PRIMITIVE BLOCKS
# ─────────────────────────────────────────────────────────────────────────────

class ConvBnRelu(nn.Module):
    """Standard Conv-BN-ReLU (used in stem, decoder fuse, MSF head)."""
    def __init__(self, in_ch, out_ch, kernel=3, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel, padding=padding, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)

print('✓ ConvBnRelu defined')

### 5b. Ghost Convolution Blocks (from ghost_convolution.ipynb)

These are the **novel components** imported from the Ghost notebook and adapted for the decoder.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# GHOST CONVOLUTION BLOCKS
#
#  GhostModule:
#    Standard convs are expensive because they compute all C_out features
#    from scratch. Ghost convolution observes that many feature maps in
#    a trained network are near-linear transformations of each other.
#    So: generate C_out/2 "intrinsic" features with a standard 3×3 conv,
#    then "hallucinate" the remaining C_out/2 features by passing the
#    intrinsic features through a cheap depthwise conv (groups=C_out/2).
#    Result: same output channels, ~50% fewer FLOPs.
#
#  ResidualGhostBlock:
#    Stack TWO GhostModules (mirrors U-Net's DoubleConv) and add a
#    residual shortcut so gradients flow unimpeded through the decoder.
#    The shortcut uses 1×1 conv + BN when channel counts differ, else Identity.
#
#  ECA_Layer (Efficient Channel Attention):
#    Global average pool → 1D conv of kernel k → sigmoid gate.
#    Captures cross-channel dependencies with ~3–5 parameters.
#    Much cheaper than SE-Net's FC bottleneck or CBAM's dual attention.
#    Applied after every decoder stage to reweight decoded channels.
# ─────────────────────────────────────────────────────────────────────────────

class GhostModule(nn.Module):
    """
    Ghost Convolution.
    Generates half the output features with a standard conv (primary_conv),
    and 'hallucinates' the other half using a cheap depthwise conv (cheap_operation).
    Output channels = out_ch (exactly), FLOPs ≈ 50% of a standard conv.
    """
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        half_ch = out_ch // 2

        # Primary path: standard 3×3 conv → half the output features
        self.primary_conv = nn.Sequential(
            nn.Conv2d(in_ch, half_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(half_ch),
            nn.ReLU(inplace=True),
        )

        # Cheap path: depthwise 3×3 conv on primary features → other half
        # groups=half_ch means each channel is processed independently (depthwise)
        self.cheap_operation = nn.Sequential(
            nn.Conv2d(half_ch, half_ch, kernel_size=3, padding=1,
                      groups=half_ch, bias=False),
            nn.BatchNorm2d(half_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.primary_conv(x)    # intrinsic features
        x2 = self.cheap_operation(x1) # ghost (hallucinated) features
        return torch.cat([x1, x2], dim=1)  # concat → out_ch channels


class ResidualGhostBlock(nn.Module):
    """
    Stacks TWO GhostModules (analogous to DoubleConv) with a residual shortcut.
    This is the drop-in replacement for DoubleConv in the RDT-UNet++ decoder.

    The residual connection:
      - Stabilises training (prevents vanishing gradients in deep decoders)
      - Allows the block to learn residual corrections rather than full mappings
      - 1×1 conv shortcut matches channel counts when in_ch ≠ out_ch
    """
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.ghost1 = GhostModule(in_ch,  out_ch)
        self.ghost2 = GhostModule(out_ch, out_ch)

        # Shortcut: match dims if needed
        self.shortcut = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_ch),
        ) if in_ch != out_ch else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        res = self.shortcut(x)   # residual branch
        x   = self.ghost1(x)
        x   = self.ghost2(x)
        return F.relu(x + res, inplace=True)  # residual fusion


class ECA_Layer(nn.Module):
    """
    Efficient Channel Attention (ECA-Net, CVPR 2020).
    1D conv on globally-pooled channel descriptor — avoids FC bottleneck.
    Adds only kernel_size (~3–5) learnable parameters per layer.
    Applied after each decoder stage to suppress irrelevant upsampled channels.
    """
    def __init__(self, k_size: int = 3):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv     = nn.Conv1d(1, 1, kernel_size=k_size,
                                  padding=(k_size - 1) // 2, bias=False)
        self.sigmoid  = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.size()
        y = self.avg_pool(x).view(b, 1, c)   # (B, 1, C)
        y = self.conv(y).view(b, c, 1, 1)    # (B, C, 1, 1)
        return x * self.sigmoid(y)            # channel-wise scaling

print('✓ GhostModule, ResidualGhostBlock, ECA_Layer defined')

### 5c. RDT-UNet++ Encoder Blocks (unchanged)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ENCODER — Dilated Residual Dense Block
#
# Three parallel dilated branches (rates 1, 2, 4) give receptive fields
# of 3×3, 5×5, 9×9 in parallel — small crater pixels survive all encoder
# levels with multi-scale context attached. Branches are summed (not concat)
# to keep output channels fixed.
# ─────────────────────────────────────────────────────────────────────────────

class DilatedConvBnRelu(nn.Module):
    def __init__(self, in_ch, out_ch, dilation=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=dilation, dilation=dilation, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class DilatedResidualDenseBlock(nn.Module):
    """
    1×1 compress → 3 parallel dilated convs (d=1,2,4) → 1×1 fuse → residual add.
    Used in all 4 encoder levels and the bottleneck.
    """
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.compress = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
        self.d1 = DilatedConvBnRelu(out_ch, out_ch, dilation=1)
        self.d2 = DilatedConvBnRelu(out_ch, out_ch, dilation=2)
        self.d4 = DilatedConvBnRelu(out_ch, out_ch, dilation=4)
        self.fuse = nn.Sequential(
            nn.Conv2d(out_ch * 3, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x   = self.compress(x)
        out = self.fuse(torch.cat([self.d1(x), self.d2(x), self.d4(x)], dim=1))
        return F.relu(out + x, inplace=True)

print('✓ DilatedResidualDenseBlock (encoder) defined')

### 5d. Skip Attention Hierarchy (unchanged)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SKIP L1, L2 — CBAM
# Channel + Spatial attention at full resolution. O(C) cost.
# ─────────────────────────────────────────────────────────────────────────────

class CBAMSkip(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.avg_pool    = nn.AdaptiveAvgPool2d(1)
        self.max_pool    = nn.AdaptiveMaxPool2d(1)
        self.channel_mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels, mid, bias=False), nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False),
        )
        self.spatial_conv = nn.Sequential(
            nn.Conv2d(2, 1, 7, padding=3, bias=False), nn.BatchNorm2d(1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C = x.shape[:2]
        ca = torch.sigmoid(
            self.channel_mlp(self.avg_pool(x)) +
            self.channel_mlp(self.max_pool(x))
        ).view(B, C, 1, 1)
        x  = x * ca
        sa = torch.sigmoid(self.spatial_conv(
            torch.cat([x.mean(dim=1, keepdim=True),
                       x.max(dim=1,  keepdim=True).values], dim=1)))
        return x * sa


# ─────────────────────────────────────────────────────────────────────────────
# SKIP L3 — Window Transformer (safe at 128×128)
# ─────────────────────────────────────────────────────────────────────────────

class WindowTransformerSkip(nn.Module):
    def __init__(self, channels: int, window_size: int = 8,
                 n_heads: int = 4, n_layers: int = 1):
        super().__init__()
        while channels % n_heads != 0 and n_heads > 1: n_heads -= 1
        self.ws = window_size
        layer = nn.TransformerEncoderLayer(
            d_model=channels, nhead=n_heads,
            dim_feedforward=channels * 2,
            dropout=0.1, batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.out_norm    = nn.LayerNorm(channels)

    def _partition(self, x, ws):
        B, C, H, W = x.shape
        pH = (ws - H % ws) % ws; pW = (ws - W % ws) % ws
        if pH > 0 or pW > 0: x = F.pad(x, (0, pW, 0, pH))
        _, _, Hp, Wp = x.shape
        x = x.reshape(B, C, Hp//ws, ws, Wp//ws, ws)
        x = x.permute(0, 2, 4, 3, 5, 1).reshape(-1, ws*ws, C)
        return x, (Hp, Wp, pH, pW)

    def _unpartition(self, x, B, C, Hp, Wp, pH, pW, ws):
        nH, nW = Hp//ws, Wp//ws
        x = x.reshape(B, nH, nW, ws, ws, C)
        x = x.permute(0, 5, 1, 3, 2, 4).reshape(B, C, Hp, Wp)
        if pH > 0 or pW > 0: x = x[:, :, :Hp-pH, :Wp-pW]
        return x

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        ws = min(self.ws, H, W)
        tokens, (Hp, Wp, pH, pW) = self._partition(x, ws)
        tokens  = self.out_norm(self.transformer(tokens))
        refined = self._unpartition(tokens, B, C, Hp, Wp, pH, pW, ws)
        return x + refined


class CrossScaleTransformerSkip(nn.Module):
    """L3 skip: window self-attn + pooled cross-attn from L4. Memory-safe."""
    def __init__(self, channels_l3: int, channels_l4: int,
                 n_heads: int = 8, n_layers: int = 1, pool_size: int = 8):
        super().__init__()
        while channels_l3 % n_heads != 0 and n_heads > 1: n_heads -= 1
        self.pool_size = pool_size
        self.l4_proj   = nn.Conv2d(channels_l4, channels_l3, 1, bias=False)
        self.self_attn = WindowTransformerSkip(channels_l3, window_size=8,
                                               n_heads=n_heads, n_layers=n_layers)
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=channels_l3, num_heads=n_heads, dropout=0.1, batch_first=True)
        self.cross_norm = nn.LayerNorm(channels_l3)
        self.expand     = nn.Conv2d(channels_l3, channels_l3, 1, bias=False)

    def forward(self, x_l3: torch.Tensor, x_l4: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x_l3.shape; ps = self.pool_size
        out  = self.self_attn(x_l3)
        l3_p = F.adaptive_avg_pool2d(out,                   (ps, ps))
        l4_p = F.adaptive_avg_pool2d(self.l4_proj(x_l4),   (ps, ps))
        q    = l3_p.flatten(2).permute(0, 2, 1)
        k    = l4_p.flatten(2).permute(0, 2, 1)
        attn_out, _ = self.cross_attn(query=q, key=k, value=k)
        q = self.cross_norm(q + attn_out)
        cross_map = q.permute(0, 2, 1).reshape(B, C, ps, ps)
        cross_up  = F.interpolate(self.expand(cross_map), size=(H, W),
                                  mode='bilinear', align_corners=True)
        return out + cross_up


# ─────────────────────────────────────────────────────────────────────────────
# SKIP L4 — Circular Transformer (window attn + radial geometry gate)
# ─────────────────────────────────────────────────────────────────────────────

class CircularTransformerSkip(nn.Module):
    """
    Window attention at L4 (64×64) + radial geometry bias via channel modulation.
    Learnable radial MLP biases attention toward concentric ring patterns (craters).
    """
    def __init__(self, channels: int, n_heads: int = 8, n_layers: int = 2,
                 window_size: int = 8):
        super().__init__()
        self.win_attn  = WindowTransformerSkip(channels, window_size=window_size,
                                               n_heads=n_heads, n_layers=n_layers)
        self.radial_mlp = nn.Sequential(
            nn.Conv2d(3, 32, 1, bias=False), nn.ReLU(inplace=True),
            nn.Conv2d(32, channels, 1, bias=False), nn.Sigmoid(),
        )

    def _radial_coords(self, H, W, device):
        ys = torch.linspace(-1, 1, H, device=device)
        xs = torch.linspace(-1, 1, W, device=device)
        gy, gx = torch.meshgrid(ys, xs, indexing='ij')
        gr = (gx**2 + gy**2).sqrt() / (2**0.5)
        return torch.stack([gx, gy, gr], dim=0).unsqueeze(0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        out    = self.win_attn(x)
        coords = self._radial_coords(H, W, x.device)
        gate   = self.radial_mlp(coords.expand(B, -1, -1, -1))
        return x + out * gate

print('✓ CBAMSkip, WindowTransformerSkip, CrossScaleTransformerSkip, CircularTransformerSkip defined')

### 5e. Ghost Decoder Block — the key new component

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# GHOST DECODER BLOCK  ← NEW (replaces RDTUpBlock's ConvBnRelu pairs)
#
#  Original RDTUpBlock decoder flow:
#    Upsample → 1×1 reduce → concat(skip) → 1×1 fuse → ConvBnRelu → Conv → residual
#
#  GhostRDTUpBlock decoder flow:
#    Upsample → 1×1 reduce → concat(skip) → 1×1 fuse → ResidualGhostBlock → ECA
#
#  Changes:
#    1. ConvBnRelu + plain Conv  →  ResidualGhostBlock
#       (Ghost conv + residual shortcut ≈ same quality, ~50% decoder FLOPs)
#    2. No internal residual (x + out) needed — ResidualGhostBlock has one built-in
#    3. ECA after the block reweights decoded channels (3–5 params)
# ─────────────────────────────────────────────────────────────────────────────
class GhostRDTUpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()

        self.fuse = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

        self.ghost_block = ResidualGhostBlock(out_ch, out_ch)
        self.eca = ECA_Layer(k_size=3)

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)

        # DEBUG SAFE CHECK
        assert x.shape[1] + skip.shape[1] == self.fuse[0].in_channels, \
            f"Expected {self.fuse[0].in_channels}, got {x.shape[1] + skip.shape[1]}"

        x = torch.cat([x, skip], dim=1)
        x = self.fuse(x)
        x = self.ghost_block(x)
        return self.eca(x)

print("✓ GhostRDTUpBlock defined")

### 5f. Multi-Scale Fusion Head & Auxiliary Heads

In [ ]:
class ChannelReducedMSFHead(nn.Module):
    """
    Channel-reduced multi-scale fusion head.
    Reduces d2, d3 channels BEFORE upsampling to save memory.
    OLD: (ch0+ch1+ch2) @ 512×512 = 448ch = ~940 MB
    NEW: (ch0+32+32)   @ 512×512 =  96ch = ~201 MB
    """
    def __init__(self, ch0: int, ch1: int, ch2: int,
                 reduce_ch: int = 32, out_channels: int = 1):
        super().__init__()
        self.reduce_d2 = nn.Sequential(
            nn.Conv2d(ch1, reduce_ch, 1, bias=False),
            nn.BatchNorm2d(reduce_ch), nn.ReLU(inplace=True),
        )
        self.reduce_d3 = nn.Sequential(
            nn.Conv2d(ch2, reduce_ch, 1, bias=False),
            nn.BatchNorm2d(reduce_ch), nn.ReLU(inplace=True),
        )
        total = ch0 + reduce_ch + reduce_ch
        self.fuse = nn.Sequential(
            ConvBnRelu(total, ch0, kernel=3, padding=1),
            ConvBnRelu(ch0, ch0 // 2, kernel=3, padding=1),
            nn.Conv2d(ch0 // 2, out_channels, 1),
        )

    def forward(self, d1, d2, d3):
        H, W  = d1.shape[2:]
        d2r   = self.reduce_d2(d2)
        d3r   = self.reduce_d3(d3)
        d2_up = F.interpolate(d2r, size=(H, W), mode='bilinear', align_corners=True)
        d3_up = F.interpolate(d3r, size=(H, W), mode='bilinear', align_corners=True)
        return self.fuse(torch.cat([d1, d2_up, d3_up], dim=1))


class AuxHead(nn.Module):
    """Auxiliary supervision head (deep supervision — discarded at inference)."""
    def __init__(self, in_ch: int, out_ch: int = 1):
        super().__init__()
        self.head = nn.Sequential(
            nn.Conv2d(in_ch, in_ch // 2, 3, padding=1, bias=False),
            nn.BatchNorm2d(in_ch // 2), nn.ReLU(inplace=True),
            nn.Conv2d(in_ch // 2, out_ch, 1),
        )

    def forward(self, x: torch.Tensor, target_size: tuple) -> torch.Tensor:
        return F.interpolate(self.head(x), size=target_size,
                             mode='bilinear', align_corners=True)

print('✓ ChannelReducedMSFHead and AuxHead defined')

### 5g. Ghost-RDT-UNet++ Full Model

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# GHOST-RDT-UNet++ — FULL MODEL
#
# Architecture Summary:
#
#  Encoder  : DilatedResidualDenseBlock (rates 1,2,4) × 4 levels
#  Skip L1  : CBAMSkip          — full res (32ch), CBAM attention
#  Skip L2  : CBAMSkip          — H/2 (64ch), CBAM attention
#  Skip L3  : CrossScaleTransformerSkip — window self-attn + pooled cross-attn from L4
#  Skip L4  : CircularTransformerSkip  — window attn + radial geometry gate
#  Bottleneck: DilatedResidualDenseBlock
#
#  Decoder  : GhostRDTUpBlock × 4   ← NEW (ResidualGhostBlock + ECA)
#  Head     : ChannelReducedMSFHead (d1 + d2_32ch + d3_32ch)
#  Aux Heads: AuxHead at d2, d3 (deep supervision, train only)
#
# Forward:
#   train mode → (main_logit, aux_l3_logit, aux_l2_logit)
#   eval mode  → main_logit
# ─────────────────────────────────────────────────────────────────────────────

class GhostRDTUNetPlusPlus(nn.Module):
    """
    Ghost-RDT-UNet++: Residual Dilated Transformer U-Net++ with Ghost Decoder.

    Combines:
      - RDT-UNet++ encoder (dilated residual dense blocks + transformer skip hierarchy)
      - Ghost Convolution decoder (ResidualGhostBlock + ECA, ~50% cheaper than standard)

    base_ch=32 → channels [32, 64, 128, 256] — memory-safe on 15 GB GPU.
    """

    def __init__(self, in_channels: int = 1, out_channels: int = 1,
                 base_ch: int = 32, attn_heads: int = 8):
        super().__init__()
        ch = [base_ch * (2 ** i) for i in range(4)]  # [32, 64, 128, 256]

        # ── Stem ──────────────────────────────────────────────────────────────
        self.stem = nn.Sequential(
            ConvBnRelu(in_channels, ch[0]),
            ConvBnRelu(ch[0], ch[0]),
        )
        self.pool = nn.MaxPool2d(2, 2)

        # ── Encoder (Dilated Residual Dense Blocks) ───────────────────────────
        # Dense connections: enc1 sees ch[0], enc2 sees ch[0],
        # enc3 sees ch[1]+ch[0] (from enc2+enc1 pooled), etc.
        self.enc1 = DilatedResidualDenseBlock(ch[0], ch[0])
        self.enc2 = DilatedResidualDenseBlock(ch[0], ch[1])
        self.enc3 = DilatedResidualDenseBlock(ch[1] + ch[0], ch[2])
        self.enc4 = DilatedResidualDenseBlock(ch[2] + ch[1] + ch[0], ch[3])

        # ── Skip Attention Hierarchy ──────────────────────────────────────────
        # self.skip1 = CBAMSkip(ch[0])         # L1: full-res CBAM
        # self.skip2 = CBAMSkip(ch[1])         # L2: H/2 CBAM
        # self.skip3 = CrossScaleTransformerSkip(
        #     ch[2], ch[3],
        #     n_heads=min(attn_heads, ch[2] // 8), n_layers=1, pool_size=8
        # )                                     # L3: window self + cross-attn from L4
        # self.skip4 = CircularTransformerSkip(
        #     ch[3],
        #     n_heads=min(attn_heads, ch[3] // 8), n_layers=2, window_size=8
        # )                                     # L4: window + radial geometry

        self.skip1 = CBAMSkip(ch[0])   # s → 32 ✅
        self.skip2 = CBAMSkip(ch[0])   # e1 → 32 ✅
        self.skip3 = CrossScaleTransformerSkip(
            ch[2], ch[3],
            n_heads=min(attn_heads, ch[2] // 8), n_layers=1, pool_size=8
        )
        self.skip4 = CircularTransformerSkip(
            ch[3],
            n_heads=min(attn_heads, ch[3] // 8), n_layers=2, window_size=8
        )
        # ── Bottleneck ────────────────────────────────────────────────────────
        self.bottleneck = DilatedResidualDenseBlock(ch[3], ch[3])

        # ── Decoder (Ghost-powered) ───────────────────────────────────────────
        # GhostRDTUpBlock replaces the original RDTUpBlock
        self.dec4 = GhostRDTUpBlock(ch[3], ch[3], ch[2])  # 256 + 256 → 128
        self.dec3 = GhostRDTUpBlock(ch[2], ch[2], ch[1])  # 128 + 128 → 64
        self.dec2 = GhostRDTUpBlock(ch[1], ch[0], ch[0])  # 64  + 32  → 32 
        self.dec1 = GhostRDTUpBlock(ch[0], ch[0], ch[0])  # 32  + 32  → 32

        # ── Multi-Scale Fusion Head ───────────────────────────────────────────
        self.head = ChannelReducedMSFHead(
            ch0=ch[0],   # d1 = 32
            ch1=ch[0],   # d2 = 32  ✅ FIX
            ch2=ch[1],   # d3 = 64  ✅ FIX
            reduce_ch=32,
            out_channels=out_channels
        )

        # ── Auxiliary Heads (deep supervision) ───────────────────────────────
        self.aux_d3 = AuxHead(ch[1], out_channels)  # from dec3
        self.aux_d2 = AuxHead(ch[0], out_channels)  # from dec2

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor):
        target_size = x.shape[2:]

        # ── Stem ──────────────────────────────────────────────────────────────
        s  = self.stem(x)        # (B, 32, H,   W)    — full resolution

        # ── Encoder with dense connections ───────────────────────────────────
        e1 = self.enc1(self.pool(s))   # (B, 32,  H/2,  W/2)
        e2 = self.enc2(self.pool(e1))  # (B, 64,  H/4,  W/4)

        # Dense connection: e3 receives e2 + downsampled e1
        e1_down = F.adaptive_avg_pool2d(e1, e2.shape[2:])
        e3 = self.enc3(self.pool(torch.cat([e2, e1_down], dim=1)))   # (B,128, H/8,  W/8)

        # Dense connection: e4 receives e3 + downsampled e2 + downsampled e1
        e2_down = F.adaptive_avg_pool2d(e2, e3.shape[2:])
        e1_down2= F.adaptive_avg_pool2d(e1, e3.shape[2:])
        e4 = self.enc4(self.pool(torch.cat([e3, e2_down, e1_down2], dim=1)))  # (B,256, H/16, W/16)

        # ── Skip Attention ────────────────────────────────────────────────────
        sk1 = self.skip1(s)            # CBAM on stem features
        sk2 = self.skip2(e1)           # CBAM on enc1 features
        sk4 = self.skip4(e4)           # Circular transformer on enc4
        sk3 = self.skip3(e3, e4)       # Cross-scale transformer: L3 queries L4

        # ── Bottleneck ────────────────────────────────────────────────────────
        bn = self.bottleneck(e4)       # (B, 256, H/16, W/16)

        # ── Ghost Decoder ─────────────────────────────────────────────────────
        d4 = self.dec4(bn,  sk4)       # (B, 128, H/8,  W/8)
        d3 = self.dec3(d4,  sk3)       # (B, 64,  H/4,  W/4)
        d2 = self.dec2(d3,  sk2)       # (B, 32,  H/2,  W/2)
        d1 = self.dec1(d2,  sk1)       # (B, 32,  H,    W)

        # ── Multi-Scale Fusion ────────────────────────────────────────────────
        main_logit = self.head(d1, d2, d3)   # (B, 1, H, W)

        if self.training:
            aux3 = self.aux_d3(d3, target_size)  # deep supervision from dec3
            aux2 = self.aux_d2(d2, target_size)  # deep supervision from dec2
            return main_logit, aux3, aux2
        return main_logit


def build_model(model_name, in_channels=1, out_channels=1, base_ch=32):
    if model_name == 'ghost_rdt_unet_plusplus':
        return GhostRDTUNetPlusPlus(in_channels, out_channels, base_ch=base_ch)
    raise ValueError(f'Unknown model: {model_name}')


# ── Parameter count sanity check ──────────────────────────────────────────────
def count_params(model):
    total  = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Total parameters    : {total:,}')
    print(f'  Trainable parameters: {trainable:,}')
    return trainable

m_test = build_model(MODEL_NAME, base_ch=32).to(DEVICE)
print(f'\n✓ {MODEL_NAME} instantiated')
count_params(m_test)

# Quick forward pass test
m_test.train()
dummy = torch.randn(1, 1, 512, 512).to(DEVICE)
with torch.no_grad():
    out = m_test(dummy)
if isinstance(out, tuple):
    print(f'  Forward pass OK — main: {out[0].shape}, aux3: {out[1].shape}, aux2: {out[2].shape}')
else:
    print(f'  Forward pass OK — output: {out.shape}')
del m_test, dummy; torch.cuda.empty_cache()
print('✓ Ghost-RDT-UNet++ defined and tested')

---
## Section 6 — Loss Functions & Metrics

In [ ]:
class DiceLoss(nn.Module):
    """Soft Dice Loss — optimises overlap directly (ideal for sparse rim masks)."""
    def __init__(self, smooth=1.0):
        super().__init__(); self.smooth = smooth
    def forward(self, logits, targets):
        p = torch.sigmoid(logits).view(-1); t = targets.view(-1)
        inter = (p * t).sum()
        return 1.0 - (2.0 * inter + self.smooth) / (p.sum() + t.sum() + self.smooth)


class DeepSupervisionLoss(nn.Module):
    """
    Handles both:
      Training  : forward(main, aux3, aux2, targets)  — 4 args
      Validation: forward(logits, targets)             — 2 args
    """
    def __init__(self, bce_weight=0.5, dice_weight=0.5, pos_weight=10.0,
                 aux_l3_weight=0.4, aux_l2_weight=0.2):
        super().__init__()
        self.aux3_w = aux_l3_weight; self.aux2_w = aux_l2_weight
        self.bw = bce_weight;        self.dw = dice_weight
        self._pw = pos_weight

    def _get_bce(self, device):
        if not hasattr(self, '_bce') or self._bce.pos_weight.device != device:
            self._bce = nn.BCEWithLogitsLoss(
                pos_weight=torch.tensor([self._pw], device=device))
        return self._bce

    def _dice(self, logits, targets, smooth=1.0):
        p = torch.sigmoid(logits).view(-1); t = targets.view(-1)
        return 1.0 - (2.0*(p*t).sum() + smooth) / (p.sum() + t.sum() + smooth)

    def _single(self, logits, targets):
        return self.bw * self._get_bce(logits.device)(logits, targets) \
             + self.dw * self._dice(logits, targets)

    def forward(self, main, aux3_or_targets, aux2=None, targets=None):
        if aux2 is None and targets is None:   # validation call
            return self._single(main, aux3_or_targets)
        # training call
        return (self._single(main, targets)
              + self.aux3_w * self._single(aux3_or_targets, targets)
              + self.aux2_w * self._single(aux2, targets))


class SegmentationMetrics:
    """Accumulates TP/FP/FN/TN and computes IoU, Dice, Precision, Recall, F1."""
    def __init__(self, threshold=0.5):
        self.threshold = threshold; self.reset()
    def reset(self):
        self.tp = self.fp = self.fn = self.tn = 0
    def update(self, logits, targets):
        preds   = (torch.sigmoid(logits) > self.threshold).float().view(-1).cpu()
        targets = targets.view(-1).cpu()
        self.tp += ((preds == 1) & (targets == 1)).sum().item()
        self.fp += ((preds == 1) & (targets == 0)).sum().item()
        self.fn += ((preds == 0) & (targets == 1)).sum().item()
        self.tn += ((preds == 0) & (targets == 0)).sum().item()
    def compute(self):
        tp, fp, fn, eps = self.tp, self.fp, self.fn, 1e-7
        iou  = tp / (tp + fp + fn + eps)
        dice = (2 * tp) / (2 * tp + fp + fn + eps)
        prec = tp / (tp + fp + eps)
        rec  = tp / (tp + fn + eps)
        f1   = (2 * prec * rec) / (prec + rec + eps)
        return {k: round(v, 4) for k, v in
                zip(['iou', 'dice', 'precision', 'recall', 'f1'],
                    [iou, dice, prec, rec, f1])}

print('✓ DeepSupervisionLoss, DiceLoss, SegmentationMetrics defined')

---
## Section 7 — Training Framework

In [ ]:
def gpu_mem_str() -> str:
    if torch.cuda.is_available():
        used  = torch.cuda.memory_allocated() / 1024**2
        total = torch.cuda.get_device_properties(0).total_memory / 1024**2
        return f"{used:.0f}/{total:.0f} MB"
    return "CPU"

def delta_str(current, previous):
    if previous is None: return ""
    diff = current - previous
    return f"({'↑' if diff >= 0 else '↓'}{abs(diff):.4f})"

def format_time(seconds):
    m, s = divmod(int(seconds), 60); return f"{m:02d}:{s:02d}"

def print_sep(char="─", w=70): print(char * w)
def print_header(title, w=70):
    print_sep("═", w); pad = (w - len(title) - 2) // 2
    print("║" + " "*pad + title + " "*(w-pad-len(title)-2) + "║")
    print_sep("═", w)


class GhostRDTTrainer:
    """
    Trainer for Ghost-RDT-UNet++.
    Handles deep supervision (3-output training mode, 1-output eval mode).
    Features: AMP, gradient clipping, CosineAnnealingLR, early stopping.
    """

    def __init__(self, model, model_name, cfg,
                 train_loader, val_loader, log_every_n_batches=10):
        self.model        = model.to(DEVICE)
        self.model_name   = model_name
        self.cfg          = cfg
        self.train_loader = train_loader
        self.val_loader   = val_loader
        self.log_every    = log_every_n_batches

        self.criterion = DeepSupervisionLoss(
            bce_weight=cfg.bce_weight, dice_weight=cfg.dice_weight,
            pos_weight=cfg.pos_weight).to(DEVICE)
        self.optimizer = AdamW(model.parameters(),
                               lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
        self.scheduler = CosineAnnealingLR(self.optimizer, T_max=cfg.epochs, eta_min=1e-6)
        self.scaler    = GradScaler(enabled=USE_AMP)
        self.metrics   = SegmentationMetrics()

        self.run_dir = Path(cfg.output_dir) / model_name
        self.run_dir.mkdir(parents=True, exist_ok=True)

        self.best_dice = 0.0; self.best_epoch = 0; self.patience = 0
        self.history   = {k: [] for k in
                          ['train_loss', 'val_loss', 'val_iou', 'val_dice',
                           'val_precision', 'val_recall', 'lr']}
        self._prev_val_dice = self._prev_val_iou = self._prev_val_loss = None

    def train_epoch(self, epoch):
        self.model.train(); total_loss = 0.0
        n_batches = len(self.train_loader); t0 = time.time()

        for batch_idx, (imgs, masks) in enumerate(self.train_loader, 1):
            imgs  = imgs.to(DEVICE, non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True)
            self.optimizer.zero_grad(set_to_none=True)

            with autocast(enabled=USE_AMP):
                main, aux3, aux2 = self.model(imgs)   # deep supervision
                loss = self.criterion(main, aux3, aux2, masks)

            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(
                self.model.parameters(), max_norm=1.0)
            self.scaler.step(self.optimizer)
            self.scaler.update()
            total_loss += loss.item()

            if batch_idx % self.log_every == 0 or batch_idx == n_batches:
                eta = (time.time()-t0)/batch_idx*(n_batches-batch_idx)
                print(f"    Batch [{batch_idx:>4d}/{n_batches}] "
                      f"{batch_idx/n_batches*100:5.1f}% | "
                      f"Loss: {loss.item():.4f} (avg: {total_loss/batch_idx:.4f}) | "
                      f"Grad: {grad_norm:.3f} | ETA: {format_time(eta)} | GPU: {gpu_mem_str()}")
        return total_loss / n_batches

    @torch.no_grad()
    def val_epoch(self):
        self.model.eval(); self.metrics.reset(); total_loss = 0.0
        print("    Validating", end="", flush=True)
        for i, (imgs, masks) in enumerate(self.val_loader):
            imgs  = imgs.to(DEVICE, non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True)
            with autocast(enabled=USE_AMP):
                logits = self.model(imgs)   # eval mode → single output
                total_loss += self.criterion(logits, masks).item()
            self.metrics.update(logits, masks)
            if i % 5 == 0: print(".", end="", flush=True)
        print(" done")
        return total_loss / len(self.val_loader), self.metrics.compute()

    def fit(self):
        n_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        print_header(f" {self.model_name.upper()} ")
        print(f"  Parameters   : {n_params:,}")
        print(f"  Device       : {DEVICE}  (AMP={USE_AMP})")
        print(f"  Epochs       : {self.cfg.epochs}")
        print(f"  Batch size   : {self.cfg.batch_size}")
        print(f"  LR           : {self.cfg.learning_rate}  (cosine → 1e-6)")
        print(f"  Loss         : BCE({self.cfg.bce_weight}) + Dice({self.cfg.dice_weight}) + DeepSup")
        print(f"  Early stop   : patience={self.cfg.early_stop_patience}")
        print_sep()

        total_start = time.time(); epoch_times = []

        for epoch in range(1, self.cfg.epochs + 1):
            t0  = time.time()
            lr_now = self.optimizer.param_groups[0]['lr']
            print(f"\n  ┌─ Epoch {epoch:03d}/{self.cfg.epochs} {'─'*30} LR: {lr_now:.2e} ─┐")
            print(f"  │  [TRAIN]")
            train_loss = self.train_epoch(epoch)
            print(f"  │  [VALID]")
            val_loss, scores = self.val_epoch()
            self.scheduler.step()

            et = time.time() - t0; epoch_times.append(et)
            eta_total = np.mean(epoch_times) * (self.cfg.epochs - epoch)

            for k, v in [('train_loss', train_loss), ('val_loss', val_loss),
                         ('val_iou', scores['iou']), ('val_dice', scores['dice']),
                         ('val_precision', scores['precision']),
                         ('val_recall', scores['recall']), ('lr', lr_now)]:
                self.history[k].append(v)

            print(f"  │")
            print(f"  │  ┌──────────────── EPOCH {epoch:03d} RESULTS ────────────────┐")
            print(f"  │  │  Train Loss : {train_loss:.6f}")
            print(f"  │  │  Val   Loss : {val_loss:.6f}   {delta_str(val_loss, self._prev_val_loss)}")
            print(f"  │  │  {'─'*45}")
            print(f"  │  │  IoU        : {scores['iou']:.6f}   {delta_str(scores['iou'], self._prev_val_iou)}")
            print(f"  │  │  Dice       : {scores['dice']:.6f}   {delta_str(scores['dice'], self._prev_val_dice)}")
            print(f"  │  │  Precision  : {scores['precision']:.6f}")
            print(f"  │  │  Recall     : {scores['recall']:.6f}")
            print(f"  │  │  F1         : {scores['f1']:.6f}")
            print(f"  │  │  {'─'*45}")
            print(f"  │  │  Epoch time : {format_time(et)}  |  ETA: {format_time(eta_total)}")
            print(f"  │  │  GPU Memory : {gpu_mem_str()}")
            print(f"  │  └─────────────────────────────────────────────────┘")

            self._prev_val_dice = scores['dice']
            self._prev_val_iou  = scores['iou']
            self._prev_val_loss = val_loss

            if scores['dice'] > self.best_dice:
                imp = scores['dice'] - self.best_dice
                self.best_dice = scores['dice']; self.best_epoch = epoch; self.patience = 0
                torch.save({'epoch': epoch, 'model_state': self.model.state_dict(),
                            'optimizer': self.optimizer.state_dict(),
                            'scaler': self.scaler.state_dict(),
                            'val_dice': self.best_dice, 'val_scores': scores,
                            'history': self.history},
                           self.run_dir / 'best_model.pth')
                print(f"  │  ✅ NEW BEST! Dice +{imp:.4f} → {self.best_dice:.4f}  [checkpoint saved]")
            else:
                self.patience += 1
                print(f"  │  ⏳ No improvement. Patience: {self.patience}/{self.cfg.early_stop_patience}")

            print(f"  └{'─'*58}┘")

            if self.patience >= self.cfg.early_stop_patience:
                print(f"\n  ⛔ Early stopping at epoch {epoch}  "
                      f"(best epoch {self.best_epoch}, Dice={self.best_dice:.4f})")
                break

        total_time = time.time() - total_start
        self._print_summary(total_time, epoch)
        with open(self.run_dir / 'history.json', 'w') as f:
            json.dump(self.history, f, indent=2)
        return {'model': self.model_name, 'best_val_dice': self.best_dice,
                'best_epoch': self.best_epoch, 'history': self.history,
                'time_min': round(total_time / 60, 2)}

    def _print_summary(self, total_time, last_epoch):
        print_sep("═")
        print(f"  TRAINING COMPLETE — {self.model_name.upper()}")
        print_sep("═")
        print(f"  Total time    : {format_time(total_time)} ({total_time/60:.1f} min)")
        print(f"  Epochs run    : {last_epoch}  |  Best epoch: {self.best_epoch}")
        print(f"  Best Val Dice : {self.best_dice:.6f}")
        chars = "▁▂▃▄▅▆▇█"
        for key, label in [('val_loss', 'Val Loss'), ('val_dice', 'Val Dice')]:
            vals = self.history[key]
            if len(vals) > 1:
                mn, mx = min(vals), max(vals) + 1e-9; rng = mx - mn
                spark = "".join(
                    chars[min(int((v-mn)/rng*(len(chars)-1)), len(chars)-1)] for v in vals[-40:])
                print(f"  {label} trend: {spark}")
        print_sep("═")
        print(f"  Checkpoint → {self.run_dir / 'best_model.pth'}")
        print(f"  History    → {self.run_dir / 'history.json'}")
        print_sep("═")

print('✓ GhostRDTTrainer defined')

---
## Section 8 — Train Ghost-RDT-UNet++

In [ ]:
print('Loading dataset...')
train_loader, val_loader, test_loader, test_indices = make_dataloaders(CFG)

In [ ]:
set_seed(CFG.seed)
print(f'\n{"#"*60}\n  TRAINING: {MODEL_NAME.upper()}\n{"#"*60}')

model = build_model(MODEL_NAME,
                    in_channels=CFG.in_channels,
                    out_channels=CFG.out_channels,
                    base_ch=32).to(DEVICE)

if torch.cuda.is_available():
    print(f'  VRAM after model load: {torch.cuda.memory_allocated()/1e9:.2f} GB')

trainer = GhostRDTTrainer(
    model, MODEL_NAME, CFG, train_loader, val_loader,
    log_every_n_batches=10
)
result = trainer.fit()

---
## Section 9 — Internal Test Set Evaluation

In [ ]:
@torch.no_grad()
def evaluate_test(model, model_name, test_loader, run_dir):
    model.eval(); metrics = SegmentationMetrics()
    for imgs, masks in tqdm(test_loader, desc=f'  Test [{model_name}]', leave=False):
        imgs  = imgs.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)
        with autocast(enabled=USE_AMP):
            logits = model(imgs)   # eval mode → single output
        metrics.update(logits, masks)
    scores = metrics.compute(); scores['model'] = model_name
    out_path = Path(run_dir) / model_name / 'test_scores.json'
    with open(out_path, 'w') as f: json.dump(scores, f, indent=2)
    print(f'  Test scores saved → {out_path}')
    return scores

# Load best checkpoint
ckpt_path = Path(CFG.output_dir) / MODEL_NAME / 'best_model.pth'
if ckpt_path.exists():
    ckpt  = torch.load(ckpt_path, map_location=DEVICE)
    state = {k.replace('_orig_mod.', ''): v for k, v in ckpt['model_state'].items()}
    model.load_state_dict(state)
    print(f'  Loaded best checkpoint (epoch {ckpt["epoch"]}, val_dice={ckpt["val_dice"]:.4f})')

test_scores = evaluate_test(model, MODEL_NAME, test_loader, CFG.output_dir)

print(f'\n{"="*50}')
print(f'  TEST RESULTS — {MODEL_NAME.upper()}')
print(f'{"="*50}')
for k, v in test_scores.items():
    if k != 'model': print(f'  {k:12s}: {v}')
print(f'{"="*50}')

---
## Section 10 — Visualisation & Analysis

In [ ]:
# ── Training Curves ───────────────────────────────────────────────────────────
def plot_training_curves(model_name=MODEL_NAME):
    hist_path = Path(CFG.output_dir) / model_name / 'history.json'
    if not hist_path.exists(): print(f'No history found for {model_name}'); return
    with open(hist_path) as f: hist = json.load(f)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'Training Curves — {model_name.upper()}', fontsize=14, fontweight='bold')
    ep = range(1, len(hist['train_loss']) + 1)

    axes[0].plot(ep, hist['train_loss'], '--', label='Train Loss', alpha=0.7)
    axes[0].plot(ep, hist['val_loss'],         label='Val Loss',   lw=2)
    axes[0].set_title('Loss (BCE + Dice + DeepSup)'); axes[0].set_xlabel('Epoch')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(ep, hist['val_dice'], label='Dice', lw=2)
    axes[1].plot(ep, hist['val_iou'],  label='IoU',  lw=2)
    axes[1].set_title('Validation Metrics'); axes[1].set_xlabel('Epoch')
    axes[1].set_ylim(0, 1); axes[1].legend(); axes[1].grid(alpha=0.3)

    axes[2].plot(ep, hist['lr'], color='orange', lw=2)
    axes[2].set_title('Learning Rate (Cosine Annealing)'); axes[2].set_xlabel('Epoch')
    axes[2].grid(alpha=0.3)

    plt.tight_layout()
    out = Path(CFG.output_dir) / model_name / 'training_curves.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    print(f'Saved → {out}')
    plt.show()

plot_training_curves()

In [ ]:
# ── Qualitative Grid: Input | GT | Prediction | Error Overlay ─────────────────
@torch.no_grad()
def predict(model, img_np):
    model.eval()
    t = torch.from_numpy(img_np.astype(np.float32)/255.0).unsqueeze(0).unsqueeze(0).to(DEVICE)
    with autocast(enabled=USE_AMP):
        out = torch.sigmoid(model(t))
    return (out.squeeze().cpu().float().numpy() > 0.5).astype(np.uint8) * 255

def overlay_masks(img, gt, pred):
    out = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR).astype(np.float32)
    gt_b, pred_b = gt > 127, pred > 127
    out[ gt_b &  pred_b] = out[ gt_b &  pred_b]*0.4 + np.array([0,   200, 0  ])*0.6  # TP green
    out[ gt_b & ~pred_b] = out[ gt_b & ~pred_b]*0.4 + np.array([0,   0,   200])*0.6  # FN blue
    out[~gt_b &  pred_b] = out[~gt_b &  pred_b]*0.4 + np.array([200, 0,   0  ])*0.6  # FP red
    return out.astype(np.uint8)

def plot_qualitative(model_name=MODEL_NAME, n_samples=5):
    ckpt_path = Path(CFG.output_dir) / model_name / 'best_model.pth'
    m = build_model(model_name, in_channels=1, out_channels=1).to(DEVICE)
    ckpt  = torch.load(ckpt_path, map_location=DEVICE)
    state = {k.replace('_orig_mod.', ''): v for k, v in ckpt['model_state'].items()}
    m.load_state_dict(state)

    img_dir  = Path(CFG.tile_image_dir); mask_dir = Path(CFG.tile_mask_dir)
    all_pngs = sorted(img_dir.glob('*.png'))
    if not all_pngs: print('No tiles found.'); return
    samples = random.sample(all_pngs, min(n_samples, len(all_pngs)))

    fig, axes = plt.subplots(n_samples, 4, figsize=(16, 4*n_samples))
    if n_samples == 1: axes = axes[np.newaxis, :]
    fig.suptitle(f'Qualitative Results — {model_name.upper()}', fontsize=15, fontweight='bold')
    for ax, t in zip(axes[0], ['Input Image', 'Ground Truth', 'Prediction', 'Error Overlay']):
        ax.set_title(t, fontsize=12, fontweight='bold')

    for i, p in enumerate(samples):
        img  = cv2.imread(str(p),               cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(str(mask_dir/p.name), cv2.IMREAD_GRAYSCALE)
        pred    = predict(m, img)
        overlay = overlay_masks(img, mask, pred)
        axes[i,0].imshow(img,  cmap='gray'); axes[i,0].axis('off')
        axes[i,1].imshow(mask, cmap='gray'); axes[i,1].axis('off')
        axes[i,2].imshow(pred, cmap='gray'); axes[i,2].axis('off')
        axes[i,3].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)); axes[i,3].axis('off')

    handles = [mpatches.Patch(color=(0, 0.78, 0),    label='True Positive'),
               mpatches.Patch(color=(0, 0,    0.78), label='False Negative (missed)'),
               mpatches.Patch(color=(0.78, 0, 0),    label='False Positive')]
    fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=10,
               bbox_to_anchor=(0.5, -0.01))
    plt.tight_layout()
    out = Path(CFG.output_dir) / model_name / 'qualitative_grid.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    print(f'Saved → {out}')
    plt.show()

plot_qualitative()

In [ ]:
# ── Crater Size Analysis: Small / Medium / Large ──────────────────────────────
@torch.no_grad()
def analyze_by_size(model_name=MODEL_NAME, n_tiles=200):
    ckpt_path = Path(CFG.output_dir) / model_name / 'best_model.pth'
    m = build_model(model_name, in_channels=1, out_channels=1).to(DEVICE)
    state = {k.replace('_orig_mod.', ''): v
             for k, v in torch.load(ckpt_path, map_location=DEVICE)['model_state'].items()}
    m.load_state_dict(state)

    img_dir  = Path(CFG.tile_image_dir); mask_dir = Path(CFG.tile_mask_dir)
    all_pngs = sorted(img_dir.glob('*.png'))
    paths    = random.sample(all_pngs, min(n_tiles, len(all_pngs)))
    bins     = {'Small\n(<200px)': [], 'Medium\n(200-2k)': [], 'Large\n(>2000px)': []}

    for p in tqdm(paths, desc=f'Size analysis [{model_name}]'):
        img  = cv2.imread(str(p),               cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(str(mask_dir/p.name), cv2.IMREAD_GRAYSCALE)
        if img is None or mask is None: continue
        pred = predict(m, img)
        _, labels, stats, _ = cv2.connectedComponentsWithStats(
            (mask > 127).astype(np.uint8), connectivity=8)
        for i in range(1, len(stats)):
            area      = stats[i, cv2.CC_STAT_AREA]
            comp_mask = (labels == i).astype(np.uint8)
            comp_pred = ((pred > 127).astype(np.uint8)) * comp_mask
            tp  = int((comp_mask & comp_pred).sum())
            fp  = int((~comp_mask.astype(bool) & comp_pred.astype(bool)).sum())
            fn  = int((comp_mask.astype(bool) & ~comp_pred.astype(bool)).sum())
            iou = tp / (tp + fp + fn + 1e-7)
            key = ('Small\n(<200px)' if area < 200 else
                   'Medium\n(200-2k)' if area < 2000 else 'Large\n(>2000px)')
            bins[key].append(iou)

    fig, ax = plt.subplots(figsize=(8, 5))
    cats   = list(bins.keys())
    means  = [np.mean(v) if v else 0 for v in bins.values()]
    counts = [len(v) for v in bins.values()]
    bars   = ax.bar(cats, means, color=['#2196F3', '#4CAF50', '#FF5722'],
                    alpha=0.85, edgecolor='black', width=0.5)
    for bar, m_val, c in zip(bars, means, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'IoU={m_val:.3f}\n(n={c})', ha='center', va='bottom', fontsize=11)
    ax.set_title(f'IoU by Crater Size — {model_name.upper()}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Mean IoU'); ax.set_ylim(0, 1.15); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    out = Path(CFG.output_dir) / model_name / 'size_analysis.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    print(f'Saved → {out}')
    plt.show()
    return {k.split('\n')[0]: round(float(np.mean(v)), 4) if v else 0 for k, v in bins.items()}

size_results = analyze_by_size()
print('Size-wise IoU:', size_results)

---
## Section 11 — External Hold-Out Test (Demo)

> Run only during the final demo with Prof. Bagade. The hold-out image was kept completely separate from training and augmentation.

In [ ]:
@torch.no_grad()
def evaluate_holdout(model_name=MODEL_NAME):
    ckpt_path = Path(CFG.output_dir) / model_name / 'best_model.pth'
    m = build_model(model_name, in_channels=CFG.in_channels,
                    out_channels=CFG.out_channels, base_ch=32).to(DEVICE)
    ckpt  = torch.load(ckpt_path, map_location=DEVICE)
    state = {k.replace('_orig_mod.', ''): v for k, v in ckpt['model_state'].items()}
    m.load_state_dict(state); m.eval()
    print(f'  Loaded checkpoint (epoch {ckpt["epoch"]}, val_dice={ckpt["val_dice"]:.4f})')

    loader  = make_holdout_loader(CFG)
    metrics = SegmentationMetrics()
    for imgs, masks in tqdm(loader, desc='Hold-out eval'):
        imgs = imgs.to(DEVICE); masks = masks.to(DEVICE)
        with autocast(enabled=USE_AMP): logits = m(imgs)
        metrics.update(logits, masks)
    scores = metrics.compute()

    print(f'\n{"="*50}\n  HOLD-OUT — {model_name.upper()}')
    print(f'  Image: {HOLDOUT_IMAGE_NAME}\n{"="*50}')
    for k, v in scores.items(): print(f'  {k:12s}: {v}')

    out = Path(CFG.output_dir) / model_name / 'holdout_scores.json'
    with open(out, 'w') as f: json.dump({**scores, 'holdout_image': HOLDOUT_IMAGE_NAME}, f, indent=2)
    print(f'  Scores saved → {out}')
    return scores


@torch.no_grad()
def visualise_holdout(model_name=MODEL_NAME, n_tiles=6):
    """Visual grid of hold-out tiles: Input | GT | Prediction | Error Overlay."""
    ckpt_path = Path(CFG.output_dir) / model_name / 'best_model.pth'
    m = build_model(model_name, in_channels=1, out_channels=1).to(DEVICE)
    state = {k.replace('_orig_mod.', ''): v
             for k, v in torch.load(ckpt_path, map_location=DEVICE)['model_state'].items()}
    m.load_state_dict(state); m.eval()

    img_dir  = Path(CFG.holdout_image_dir)
    mask_dir = Path(CFG.holdout_mask_dir)
    all_pngs = sorted(img_dir.glob('*.png'))
    samples  = random.sample(all_pngs, min(n_tiles, len(all_pngs)))

    fig, axes = plt.subplots(len(samples), 4, figsize=(16, 4*len(samples)))
    if len(samples) == 1: axes = axes[np.newaxis, :]
    fig.suptitle(f'Hold-Out Demo — {model_name.upper()}\n{HOLDOUT_IMAGE_NAME}',
                 fontsize=14, fontweight='bold')
    for ax, t in zip(axes[0], ['Input', 'Ground Truth', 'Prediction', 'Error Overlay']):
        ax.set_title(t, fontsize=11, fontweight='bold')

    for i, p in enumerate(samples):
        img  = cv2.imread(str(p),               cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(str(mask_dir/p.name), cv2.IMREAD_GRAYSCALE)
        pred    = predict(m, img)
        overlay = overlay_masks(img, mask, pred)
        axes[i,0].imshow(img,  cmap='gray'); axes[i,0].axis('off')
        axes[i,1].imshow(mask, cmap='gray'); axes[i,1].axis('off')
        axes[i,2].imshow(pred, cmap='gray'); axes[i,2].axis('off')
        axes[i,3].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)); axes[i,3].axis('off')

    handles = [mpatches.Patch(color=(0, 0.78, 0),    label='True Positive'),
               mpatches.Patch(color=(0, 0,    0.78), label='False Negative'),
               mpatches.Patch(color=(0.78, 0, 0),    label='False Positive')]
    fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=10,
               bbox_to_anchor=(0.5, -0.01))
    plt.tight_layout()
    out = Path(CFG.output_dir) / model_name / 'holdout_visual.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    print(f'Saved → {out}')
    plt.show()


# ▶ Run hold-out evaluation
holdout_scores = evaluate_holdout()
visualise_holdout(n_tiles=6)